# Data Validation - Combined Cycle Power Plant

Tahap ini bertujuan melakukan validasi awal terhadap dataset sebelum
dilakukan proses cleaning dan preprocessing.

Validasi yang dilakukan meliputi:
1. Pemeriksaan struktur data
2. Pemeriksaan tipe data
3. Pemeriksaan missing value
4. Pemeriksaan duplicate
5. Pemeriksaan rentang nilai
6. Pemeriksaan nilai unik
7. Pemeriksaan konsistensi antar-sheet

In [2]:
import pandas as pd
import numpy as np

In [3]:
file_path = "../data/raw/Folds5x2_pp.xlsx"

df = pd.read_excel(
    file_path,
    sheet_name="Sheet1"
)

df.head()

,AT,V,AP,RH,PE
0,14.96,41.76,1024.07,73.17,463.26
1,25.18,62.96,1020.04,59.08,444.37
2,5.11,39.40,1012.16,92.14,488.56
3,20.86,57.32,1010.24,76.64,446.48
4,10.82,37.50,1009.23,96.62,473.90


## 1. Struktur Dataset

In [4]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)

Shape: (9568, 5)
Columns: ['AT', 'V', 'AP', 'RH', 'PE']

Data types:
AT    float64
V     float64
AP    float64
RH    float64
PE    float64
dtype: object


## 2. Missing Value

In [5]:
missing = df.isnull().sum()

print(missing)
print("\nTotal missing values:", missing.sum())

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

Total missing values: 0


## 3. Duplicate Data

In [6]:
duplicate_count = df.duplicated().sum()

print("Jumlah duplicate:", duplicate_count)

Jumlah duplicate: 41


In [7]:
duplicate_rows = df[df.duplicated(keep=False)]

duplicate_rows.head(10)

,AT,V,AP,RH,PE
231,26.85,75.60,1017.43,74.55,433.48
345,12.19,41.17,1019.43,65.47,475.64
352,25.56,75.60,1017.37,69.31,442.20
468,25.21,75.60,1017.19,69.84,445.84
726,28.41,75.60,1018.48,56.07,440.28
806,6.86,41.17,1020.12,79.14,490.23
1186,6.06,41.17,1019.67,84.70,489.62
1291,27.54,75.60,1018.26,57.89,440.87
1482,12.11,41.17,1019.46,62.59,475.53
1551,8.57,41.17,1020.18,72.47,484.20


In [8]:
duplicate_rows.shape

(82, 5)

In [9]:
summary = df.describe().T

summary

,count,mean,std,min,25%,50%,75%,max
AT,9568.0,19.651231,7.452473,1.81,13.5100,20.345,25.72,37.11
V,9568.0,54.305804,12.707893,25.36,41.7400,52.080,66.54,81.56
AP,9568.0,1013.259078,5.938784,992.89,1009.1000,1012.940,1017.26,1033.30
RH,9568.0,73.308978,14.600269,25.56,63.3275,74.975,84.83,100.16
PE,9568.0,454.365009,17.066995,420.26,439.7500,451.550,468.43,495.76


memastikan nilai masuk akal secara numerik dan tidak terdapat angka yang jelas rusak seperti NaN, inf, atau tipe string

In [11]:
np.isinf(df).sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [12]:
np.isinf(df).sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [13]:
for column in df.columns:
    print(f"{column}: {df[column].nunique()} unique values")

AT: 2773 unique values
V: 634 unique values
AP: 2517 unique values
RH: 4546 unique values
PE: 4836 unique values


In [14]:
constant_columns = [
    column for column in df.columns
    if df[column].nunique() <= 1
]

constant_columns

[]

untuk mengetahui apakah ada kolom yang aneh atau konstan.

In [15]:
all_sheets = pd.read_excel(
    file_path,
    sheet_name=None
)

for sheet_name, sheet_df in all_sheets.items():
    print(
        sheet_name,
        "| shape:", sheet_df.shape,
        "| columns:", sheet_df.columns.tolist()
    )

Sheet1 | shape: (9568, 5) | columns: ['AT', 'V', 'AP', 'RH', 'PE']
Sheet2 | shape: (9568, 5) | columns: ['AT', 'V', 'AP', 'RH', 'PE']
Sheet3 | shape: (9568, 5) | columns: ['AT', 'V', 'AP', 'RH', 'PE']
Sheet4 | shape: (9568, 5) | columns: ['AT', 'V', 'AP', 'RH', 'PE']
Sheet5 | shape: (9568, 5) | columns: ['AT', 'V', 'AP', 'RH', 'PE']


In [16]:
sheet_names = list(all_sheets.keys())

base_df = all_sheets[sheet_names[0]].sort_values(
    by=list(all_sheets[sheet_names[0]].columns)
).reset_index(drop=True)

for sheet_name in sheet_names[1:]:
    current_df = all_sheets[sheet_name].sort_values(
        by=list(all_sheets[sheet_name].columns)
    ).reset_index(drop=True)

    print(
        sheet_name,
        "same as Sheet1:",
        base_df.equals(current_df)
    )

Sheet2 same as Sheet1: True
Sheet3 same as Sheet1: True
Sheet4 same as Sheet1: True
Sheet5 same as Sheet1: True


In [17]:
validation_summary = {
    "rows": df.shape[0],
    "columns": df.shape[1],
    "missing_values": int(df.isnull().sum().sum()),
    "duplicate_rows": int(df.duplicated().sum()),
    "infinite_values": int(np.isinf(df).sum().sum()),
    "negative_values": int((df < 0).sum().sum()),
    "constant_columns": len(constant_columns),
}

validation_summary

{'rows': 9568,
 'columns': 5,
 'missing_values': 0,
 'duplicate_rows': 41,
 'infinite_values': 0,
 'negative_values': 0,
 'constant_columns': 0}

## Kesimpulan Validasi

Berdasarkan validasi awal pada Sheet1, dataset memiliki 9.568 observasi
dan 5 variabel numerik, yaitu AT, V, AP, RH, dan PE.

Hasil validasi menunjukkan:
- Tidak terdapat missing value.
- Tidak terdapat nilai infinite.
- Tidak terdapat nilai negatif.
- Tidak terdapat kolom konstan.
- Terdapat 41 baris yang teridentifikasi sebagai duplicate.

Kelima sheet memiliki struktur dan isi data yang konsisten setelah
urutan baris diabaikan. Oleh karena itu, Sheet1 digunakan sebagai
dataset utama untuk tahap pengolahan berikutnya.

Duplicate belum dihapus pada tahap ini karena keputusan cleaning akan
dilakukan pada tahap data cleaning setelah karakteristik data dianalisis
lebih lanjut.